# 02 - Species classification (M2-M3)
Three variants under one config and seed: a CNN trained from scratch, ResNet-50 with frozen features,
and ResNet-50 fully fine-tuned. Model selection uses validation macro-F1 only; the test split is read
once at the end. Temperature scaling is fitted on validation to calibrate the confidence that the
system's refer-to-staff rule relies on.

In [ ]:
# Setup — mount Drive before resolving the repository.
import os, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")

def _resolve_spv_repo(search_root: Path, explicit: str | None = None) -> Path:
    def valid(path):
        return (path / "src" / "spv").is_dir() and (path / "configs" / "config.yaml").is_file()
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if not valid(candidate):
            raise FileNotFoundError(f"SPV_REPO is not a project root: {candidate}")
        return candidate
    candidates = set()
    for pattern in ("smart-pet-vision", "*smart-pet-vision", "*/smart-pet-vision", "*/*/smart-pet-vision"):
        candidates.update(path.resolve() for path in search_root.glob(pattern) if valid(path))
    if valid(search_root):
        candidates.add(search_root.resolve())
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one smart-pet-vision repository under {search_root}; found {len(candidates)}. "
            "Add the shared folder shortcut to My Drive, or set SPV_REPO to its exact path."
        )
    return candidates.pop()

if IN_COLAB:
    REPO = _resolve_spv_repo(Path("/content/drive/MyDrive"), os.environ.get("SPV_REPO"))
else:
    explicit = os.environ.get("SPV_REPO")
    local_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                       if (path / "src" / "spv").is_dir()), Path.cwd())
    REPO = _resolve_spv_repo(local_root, explicit)
os.chdir(REPO)
if IN_COLAB:
    !pip -q install -r requirements.txt
sys.path.insert(0, os.path.abspath("src"))

from spv.common import load_config, seed_everything, save_json, load_json
cfg = load_config()
if IN_COLAB:  # images on fast local disk (re-downloaded each session); results stay on Drive
    cfg["paths"]["raw"] = Path("/content/data/raw")
    cfg["paths"]["processed"] = Path("/content/data/processed")
seed_everything(cfg["seed"])
RUNS = Path(cfg["paths"]["runs"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from spv import data
df = data.ensure_data(cfg)          # first run: download + clean + split (about 10 min)
IMG = data.dataset_dir(cfg) / "images"
PROC = Path(cfg["paths"]["processed"])
print(len(df), "images after cleaning")

In [ ]:
from spv import classify
OUT = RUNS / "classification"
loaders = classify.loaders(df, IMG, cfg)
train_info = []
for kind in ("scratch", "frozen", "finetune"):
    train_info.append(classify.train(kind, loaders, cfg, OUT))
save_json(train_info, OUT / "train_info.json"); train_info

## Learning curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
(OUT / "figures").mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for kind in ("scratch", "frozen", "finetune"):
    h = pd.read_csv(OUT / f"{kind}_history.csv")
    ax[0].plot(h.epoch, h.train_loss, label=kind); ax[1].plot(h.epoch, h.val_macro_f1, label=kind)
ax[0].set(title="Training loss", xlabel="epoch"); ax[1].set(title="Validation macro-F1", xlabel="epoch")
ax[1].legend(); plt.tight_layout(); fig.savefig(OUT / "figures" / "learning_curves.png", dpi=150)

## Test evaluation (read once)

In [ ]:
results = []
for kind in ("scratch", "frozen", "finetune"):
    model = classify.load_model(kind, cfg, OUT / f"{kind}.pt")
    results.append(classify.run_evaluation(kind, model, loaders, df, IMG, OUT))
table = classify.comparison_table(results, train_info)
table.to_csv(OUT / "comparison.csv", index=False); table.round(4)

In [ ]:
# model selection uses validation only; the test split is reported, never used to choose
best = table.sort_values("val_macro_f1", ascending=False).iloc[0]
print("Best variant (by validation macro-F1):", best.model)
r = next(x for x in results if x["kind"] == best.model)["test_calibrated"]
print("Per class:", r["per_class"])
pd.DataFrame(r["by_breed"]).head(10)

In [ ]:
from IPython.display import Image as Show, display
for f in ("confusion", "reliability_raw", "reliability_cal"):
    display(Show(str(OUT / "figures" / f"{best.model}_{f}.png"), width=380))
display(Show(str(OUT / "figures" / f"{best.model}_errors.jpg"), width=900))

## Error analysis and data-improvement plan
Scratch CNN makes 104 test errors; the selected fine-tuned model makes one (Bombay cat), and the frozen model makes one (Sphynx cat). Fur appearance and pose are possible contributors, not proven causes. The random species-label audit does not verify these particular images.
Collect new examples covering harder poses, young animals, occlusion and under-represented cat appearances. Fit calibration on a separate harder set and compare alternative strategies, including a T >= 1 constraint as a hypothesis. Keep both existing test sets untouched.
